# 00 - Separación de datos Train y Test

## Telco Customer Churn

### Objetivo
Separar el conjunto de datos original en dos subconjuntos antes del análisis exploratorio y del proceso ETL:

- **Train (80%)**: utilizado para EDA, definición de transformaciones, preparación y entrenamiento de modelos.
- **Test (20%)**: reservado y no utilizado durante EDA, ETL de entrenamiento ni ajuste de modelos. Se utilizará posteriormente para la evaluación final.

Se utilizará una separación estratificada respecto de `Churn` para mantener una proporción similar de clientes que abandonaron y que permanecieron en ambos conjuntos.

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

## 1. Configuración de rutas

Las rutas se construyen a partir de la ubicación del notebook para que el proyecto sea reproducible sin depender de una ruta absoluta del computador.

In [ ]:
BASE_DIR = Path.cwd().parent

RAW_PATH = BASE_DIR / "data" / "raw" / "Telco_Customer_Churn_Dataset.csv"
SPLIT_DIR = BASE_DIR / "data" / "split"

SPLIT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = SPLIT_DIR / "telco_train.csv"
TEST_PATH = SPLIT_DIR / "telco_test.csv"

print(f"Dataset original: {RAW_PATH}")
print(f"Train: {TRAIN_PATH}")
print(f"Test:  {TEST_PATH}")

## 2. Carga del dataset original

In [ ]:
df = pd.read_csv(RAW_PATH)

print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
df.head()

## 3. Definición de variables

`Churn` es la variable objetivo del problema de clasificación.

In [ ]:
TARGET = "Churn"

X = df.drop(columns=[TARGET])
y = df[TARGET]

## 4. Separación Train/Test

Se utiliza un 80% de los datos para entrenamiento y un 20% para prueba.

La opción `stratify=y` mantiene aproximadamente la misma distribución de `Churn` en ambos conjuntos.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

train = X_train.copy()
train[TARGET] = y_train

test = X_test.copy()
test[TARGET] = y_test

print(f"Train: {train.shape[0]} filas")
print(f"Test:  {test.shape[0]} filas")

## 5. Verificación de la distribución de Churn

Esta comprobación permite confirmar que la separación estratificada conservó aproximadamente la proporción de la variable objetivo.

In [ ]:
comparacion_churn = pd.DataFrame({
    "Original (%)": y.value_counts(normalize=True).mul(100),
    "Train (%)": y_train.value_counts(normalize=True).mul(100),
    "Test (%)": y_test.value_counts(normalize=True).mul(100)
})

comparacion_churn

## 6. Verificación de duplicados entre Train y Test

Se comprueba que los identificadores de clientes no aparezcan simultáneamente en ambos conjuntos.

In [ ]:
ids_train = set(train["customerID"])
ids_test = set(test["customerID"])
ids_compartidos = ids_train.intersection(ids_test)

print(f"Clientes compartidos entre Train y Test: {len(ids_compartidos)}")

## 7. Guardado de los conjuntos

Los archivos se guardan sin realizar limpieza, imputación, codificación ni escalamiento. Es importante mantener esta etapa separada del ETL.

In [ ]:
train.to_csv(TRAIN_PATH, index=False)
test.to_csv(TEST_PATH, index=False)

print("Separación completada correctamente.")
print(f"Train guardado en: {TRAIN_PATH}")
print(f"Test guardado en:  {TEST_PATH}")

## 8. Regla de trabajo posterior

A partir de este punto:

1. `telco_train.csv` será utilizado para el EDA y para definir las transformaciones.
2. `telco_test.csv` permanecerá reservado para la evaluación final.
3. Las transformaciones que dependan de parámetros calculados con los datos deberán aprenderse exclusivamente con Train y posteriormente aplicarse a Test.
4. No se debe utilizar información de Test para tomar decisiones sobre variables, imputaciones, escalamiento, selección de características o hiperparámetros.
5. El notebook `01_EDA.ipynb` deberá leer `telco_train.csv`, no el dataset completo original.